In [1]:
import torch, math
import h5py
import numpy as np
import os
import matplotlib.pyplot as plt
import sys
from models import LCSONet
ROOT = os.getcwd()
sys.path.insert(0, os.path.join(ROOT, "tests"))
sys.path.insert(0, ROOT)

In [2]:
from trainer import train_hits_classifier
from models import DeepONetClassifier, LCSONet
import utils

In [3]:
DEVICE = torch.device("cuda")
OUT   = "outputs/"
DTYPE = torch.float64

In [4]:
h5_path = os.path.join('outputs', 'training_data.h5')
with h5py.File(h5_path, 'r') as h5f:
        phis_ds = h5f['phis']
        muons_ds = h5f['muons']
        hits_ds = h5f['hits']
        phis_training = torch.from_numpy(phis_ds[:]).to(DTYPE)
        hits = torch.from_numpy((hits_ds[:]).sum(axis=1, keepdims=True)).to(DTYPE)
        print(f"Loaded phis {phis_ds.shape}, muons {muons_ds.shape}, hits {hits_ds.shape} from {h5_path}")
phi_0 = phis_training[0,...].reshape(1, phis_training.shape[-1])
phi_dim = phis_training.shape[-1]

Loaded phis (257, 43), muons (257, 5000000, 7), hits (257, 5000000) from outputs/training_data.h5


In [5]:
from botorch.models import SingleTaskGP
from botorch.models.transforms.outcome import Standardize
from botorch.fit import fit_gpytorch_mll
from gpytorch.kernels import MaternKernel, ScaleKernel
from gpytorch.means import ConstantMean
from gpytorch.mlls import ExactMarginalLogLikelihood
from gpytorch.priors import LogNormalPrior
from botorch.models.transforms.input import Normalize

In [6]:
tau = torch.tensor(0.2, dtype=DTYPE)
U = ((phis_training.to(DTYPE) - phi_0.to(DTYPE)) / tau.to(DTYPE))   # scaled inputs
Y = hits.to(DTYPE).reshape(-1, 1)     # (n, 1), not (n,)

assert U.dim() == 2 and Y.dim() == 2 and U.shape[0] == Y.shape[0], (U.shape, Y.shape)


In [7]:
d = U.shape[-1]

In [8]:
loc  = math.sqrt(2.0) + 0.5 * math.log(d)
base = MaternKernel(nu=2.5, ard_num_dims=d,
                    lengthscale_prior=LogNormalPrior(loc, math.sqrt(3.0)))
base.lengthscale = math.exp(loc)
covar = ScaleKernel(base, outputscale_prior=LogNormalPrior(0.0, 1.0))
covar.outputscale = 1.0

model = SingleTaskGP(
    train_X=U,
    train_Y=Y, 
    mean_module=ConstantMean(),
    covar_module=covar,
    input_transform=Normalize(d=d, bounds=torch.stack([U.min(0).values,
                                                       U.max(0).values])),
    outcome_transform=Standardize(m=1),
).to(DTYPE)

fit_gpytorch_mll(ExactMarginalLogLikelihood(model.likelihood, model))
model.eval()


SingleTaskGP(
  (likelihood): GaussianLikelihood(
    (noise_covar): HomoskedasticNoise(
      (noise_prior): LogNormalPrior()
      (raw_noise_constraint): GreaterThan(1.000E-04)
    )
  )
  (mean_module): ConstantMean()
  (covar_module): ScaleKernel(
    (base_kernel): MaternKernel(
      (lengthscale_prior): LogNormalPrior()
      (raw_lengthscale_constraint): Positive()
    )
    (outputscale_prior): LogNormalPrior()
    (raw_outputscale_constraint): Positive()
  )
  (outcome_transform): Standardize()
  (input_transform): Normalize()
)

In [9]:
U_new = ((phis_training.to(DTYPE) - phi_0.to(DTYPE)) / tau.to(DTYPE))   # scaled inputs
with torch.no_grad():
    post = model.posterior(U_new)                # U_new in scaled units
    mu, var = post.mean.squeeze(-1), post.variance.squeeze(-1)   # in hits

In [10]:
with torch.no_grad():
    Ut = model.input_transform(U)
    nz = model.likelihood.noise
    nz = nz.expand(len(U)) if nz.numel() == 1 else nz
    A  = torch.cholesky_inverse(torch.linalg.cholesky(
             model.covar_module(Ut).to_dense() + torch.diag_embed(nz)))
    r, dA = model.train_targets - model.mean_module(Ut), torch.diagonal(A)
    z = model.train_targets
    print("Q2", (1 - ((A @ r) / dA).pow(2).sum() / (z - z.mean()).pow(2).sum()).item())

Q2 0.9710985575605433
